### Install kaggle API

In [ ]:
!pip install kaggle

### Upload Kaggle.jason

In [ ]:
from google.colab import files
files.upload()

### Configure Kaggle

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

### Download ASL Dataset

In [ ]:
!kaggle datasets download -d grassknoted/asl-alphabet
!unzip asl-alphabet.zip

### Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

### Create Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    '/content/asl_alphabet_train/asl_alphabet_train',
    target_size=(64,64),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    '/content/asl_alphabet_train/asl_alphabet_train',
    target_size=(64,64),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

### Build Strong CNN Model

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(29, activation='softmax')  # 29 classes
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Train Model

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

### Evaluate Model

In [ ]:
loss,acc =model.evaluate(val_generator)
print("Final Accuracy:",acc*100)

In [ ]:
model.save("asl_model.keras")

### Final Accuracy

In [ ]:
print("Final Training Accuracy:", history.history['accuracy'][-1] * 100)
print("Final Validation Accuracy:", history.history['val_accuracy'][-1] * 100)

Accuracy for all epochs

In [ ]:
print(history.history['accuracy'])
print(history.history['val_accuracy'])

### Plot Accuracy Graph

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])
plt.show()